# PA = LU Decomposition (with Partial Pivoting)

### Why is it used?
Standard LU decomposition (and standard Gaussian elimination) fails catastrophically if it encounters a zero on the diagonal (a zero pivot) because you cannot divide by zero. Furthermore, even if the pivot isn't zero, a *very small* pivot can cause massive numerical instability due to round-off errors blowing up when dividing by tiny numbers.

**PA = LU decomposition** solves this by using *partial pivoting*. Before processing each column, it searches for the largest absolute value in that column and swaps that row to the top. This guarantees we always divide by the largest possible number, ensuring maximum numerical stability. 

The $P$ is a Permutation matrix that simply keeps track of all the row swaps we made, so that the math $PA = LU$ still balances perfectly.


In [ ]:
import numpy as np

def pa_lu_decomposition(A):
    n = A.shape[0]
    L = np.eye(n)
    U = A.copy()
    P = np.eye(n)
    
    for i in range(n):
        # Partial Pivoting: find the row with the largest absolute value in the current column
        max_idx = np.argmax(np.abs(U[i:, i])) + i
        
        # Swap rows in U, P, and the previously calculated parts of L
        if i != max_idx:
            # Swap in U
            U[i], U[max_idx] = U[max_idx].copy(), U[i].copy()
            # Swap in P
            P[i], P[max_idx] = P[max_idx].copy(), P[i].copy()
            # Swap in L (only the columns up to i)
            if i > 0:
                L[i, :i], L[max_idx, :i] = L[max_idx, :i].copy(), L[i, :i].copy()
                
        # Standard LU elimination steps
        for j in range(i+1, n):
            factor = U[j, i] / U[i, i]
            L[j, i] = factor
            U[j] -= factor * U[i]
            
    return P, L, U

def solve_pa_lu(P, L, U, b):
    # 1. Apply permutation to b (We are solving L U x = P b)
    Pb = P @ b
    
    # 2. Forward substitution (Solve L y = Pb)
    y = np.linalg.solve(L, Pb)
        
    # 3. Backward substitution (Solve U x = y)
    x = np.linalg.solve(U, y)
        
    return x

# Example using a matrix that would fail standard LU (0 on diagonal)
A = np.array([[0.0, 2.0, 3.0], [1.0, -1.0, 1.0], [-1.0, 1.0, 0.0]])
b = np.array([4.0, 3.0, 1.0])

# Factorize once
P, L, U = pa_lu_decomposition(A)

# Solve for x
x = solve_pa_lu(P, L, U, b)

print("P matrix:\n", P)
print("\nL matrix:\n", L)
print("\nU matrix:\n", U)
print("\nVerify P @ A == L @ U:\n", np.allclose(P @ A, L @ U))
print("\nSolution x:", x)
